# Notebook 04: Deploy a real-time endpoint and monitor it

Deploy the version approved in notebook 03 and monitor it. Nothing is trained here.

|||
|---|---|
|00. Environment setup ||
|01. Experiment in a notebook, then scale with SageMaker AI jobs ||
|02. Automate model governance with EventBridge and Model Cards ||
|03. Operationalize with an ML pipeline and model registry ||
|**04. Deploy to an endpoint and monitor** |**◄ YOU ARE HERE**|
|05. Templatize staged deployment as a Project ||

![Approved version deployed with data capture; Evidently compares captured traffic to the baseline and to ground truth; results go to MLflow](img/monitoring-flow-aws.svg)

## What you do here

1. Resolve the latest version and deploy only if it is `production` **and** `Approved`.
2. Test the logged `inference.py` in the kernel before paying for an endpoint.
3. Deploy with [data capture](https://docs.aws.amazon.com/sagemaker/latest/dg/model-monitor-data-capture.html) and test the endpoint.
4. Send traffic with an `InferenceId` per request.
5. Check data drift and model quality with [Evidently](https://www.evidentlyai.com/evidently-oss) and log them to MLflow.

**Outcome:** an approved model serving live traffic from a real-time endpoint, with data capture, and data drift and model quality checks on its traffic logged to MLflow.

<div style="border: 2px solid #ff9900; border-radius: 8px; padding: 15px; background-color: #fff3e0; margin: 10px 0;">
<strong>⚠️ Simulated traffic.</strong> The production traffic is the test set with an artificial shift on three features, so drift detection has something to find.
</div>

In [ ]:
import sys

assert sys.version_info[:2] == (3, 12), f"Expected Python 3.12, got {sys.version_info.major}.{sys.version_info.minor}. Please make sure you run on Python 3.12"

## Setup

Restore the variables stored by notebooks 00–03.

In [ ]:
%store -r

try:
    initialized
except NameError:
    print("+++++++++++++++++++++++++++++++++++++++++++++++++")
    print("[ERROR] YOU HAVE TO RUN 00-start-here notebook   ")
    print("+++++++++++++++++++++++++++++++++++++++++++++++++")

# Everything below is produced by notebooks 01 and 03 and restored by %store -r.
# The datasets are header-less CSVs, so the feature names travel as a stored variable.
required = ["registered_model_name", "experiment_name", "test_s3_url", "baseline_s3_url", "feature_names", "xgboost_image"]
missing = [name for name in required if name not in dir()]
if missing:
    raise NameError(
        f"Missing stored variables {missing}. Run notebooks 01 and 03 first; they %store these values."
    )

In [ ]:
import os
import json
import time
import base64
import importlib.util
from datetime import datetime

import boto3
import numpy as np
import pandas as pd
import mlflow
from mlflow import MlflowClient
from IPython.display import HTML, IFrame, display
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, confusion_matrix

from sagemaker.core.resources import Model, EndpointConfig
from sagemaker.core.shapes import (
    ContainerDefinition,
    ProductionVariant,
    DataCaptureConfig,
    CaptureOption,
    CaptureContentTypeHeader,
)
from sagemaker.core.s3 import S3Downloader
from sagemaker.core.helper.session_helper import Session

from evidently import Report, Dataset, DataDefinition, BinaryClassification
from evidently.presets import DataDriftPreset, DataSummaryPreset, ClassificationPreset

# Registry helpers shared with notebooks 01 and 03.
from governance.approval_gate import latest_model_package, print_lifecycle, mlflow_version_for_package

In [ ]:
sagemaker_session = Session()
sm_client = boto3.client("sagemaker", region_name=region)
runtime_client = boto3.client("sagemaker-runtime", region_name=region)
s3_client = boto3.client("s3", region_name=region)

# MLflow prints "View run ... at: https://mlflow.sagemaker.<region>.app.aws/..." when a run
# ends. That host is the SageMaker MLflow *tracking API* endpoint, not the UI, so the links
# do not open. Suppress them; the notebook shows working presigned UI links instead.
os.environ["MLFLOW_SUPPRESS_PRINTING_URL_TO_STDOUT"] = "true"
mlflow.set_tracking_uri(mlflow_arn)
mlflow_client = MlflowClient()
experiment = mlflow.set_experiment(experiment_name)

print("Configuration:")
print(f"  Region:            {region}")
print(f"  S3 bucket/prefix:  s3://{bucket_name}/{bucket_prefix}")
print(f"  IAM role:          {sm_role}")
print(f"  MLflow experiment: {experiment_name}")
print(f"  Registered model:  {registered_model_name}")
print(f"  Features:          {len(feature_names)} (e.g. {feature_names[:4]})")

---
## 1. Resolve the deployment candidate

`latest_model_package()` finds the group the sync created and returns its latest version. The cell **refuses to deploy** anything that is not `production` and `Approved`. The registry state is the contract.

It also prints the inference specification (image, model data, `SAGEMAKER_PROGRAM`) and the provenance on the version.

In [ ]:
latest = latest_model_package(sm_client, registered_model_name, prefix="aim408-from-idea-to-prod-model-")
model_package_arn = latest["ModelPackageArn"]

state = print_lifecycle(sm_client, model_package_arn, "Latest version in the registry:")
if not (state["stage"] == "production" and state["approval_status"] == "Approved"):
    raise RuntimeError(
        f"Refusing to deploy {model_package_arn}: stage={state['stage']}, "
        f"approval={state['approval_status']}. Only production/Approved versions are deployed. "
        "Finish the approval gate in notebook 03 first."
    )

described = sm_client.describe_model_package(ModelPackageName=model_package_arn)
container = described["InferenceSpecification"]["Containers"][0]
provenance = described.get("CustomerMetadataProperties", {})

print("\nInference specification (what the endpoint will run):")
print(f"  image:              {container['Image']}")
print(f"  model data source:  {container['ModelDataSource']['S3DataSource']['S3Uri']}")
print(f"  environment:        {container.get('Environment', {})}")
print(f"  instance types:     {described['InferenceSpecification']['SupportedRealtimeInferenceInstanceTypes']}")
print(f"  content types:      {described['InferenceSpecification']['SupportedContentTypes']}")

print("\nProvenance recorded on the version:")
for key in ("RegistrationSource", "RegisteredBy", "Pipeline", "TestAuc", "ApprovedBy", "ApprovedAt"):
    if key in provenance:
        print(f"  {key:<20}{provenance[key]}")
print(f"\nModel card status:    {described.get('ModelCard', {}).get('ModelCardStatus', '(no card)')}")

---
## 2. Validate the serving code in this kernel

The endpoint downloads the model's S3 prefix to `/opt/ml/model` and calls the logged `inference.py` handlers. Do the same here: no Docker, no endpoint, no cost. A failure here would otherwise appear as a `/ping` timeout minutes into provisioning.

### Prepare the test data

In [ ]:
os.makedirs("tmp", exist_ok=True)
S3Downloader.download(f"{test_s3_url}/test_x.csv", "tmp")
S3Downloader.download(f"{test_s3_url}/test_y.csv", "tmp")

test_x = pd.read_csv("tmp/test_x.csv", header=None, names=feature_names)
test_y = pd.read_csv("tmp/test_y.csv", header=None).iloc[:, 0].to_numpy()
print(f"test_x: {test_x.shape} | test_y: {test_y.shape}")
test_x.head(3)

### Download what the endpoint downloads

In [ ]:
model_data_s3_uri = container["ModelDataSource"]["S3DataSource"]["S3Uri"]
local_model_dir = os.path.abspath("tmp/model")
os.makedirs(local_model_dir, exist_ok=True)
S3Downloader.download(model_data_s3_uri, local_model_dir)

print(f"Model data source: {model_data_s3_uri}")
print(f"Downloaded to:     {local_model_dir}")
print(f"Contents:          {sorted(os.listdir(local_model_dir))}")
print(f"code/:             {sorted(os.listdir(os.path.join(local_model_dir, 'code')))}")

### Import the script and call the handlers

Load the script named by `SAGEMAKER_PROGRAM`, call `model_fn`, then send one CSV request through `input_fn` → `predict_fn` → `output_fn`.

In [ ]:
program = container["Environment"]["SAGEMAKER_PROGRAM"]
script_path = os.path.join(local_model_dir, "code", program)

spec = importlib.util.spec_from_file_location("user_inference_module", script_path)
inference_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(inference_module)

local_model = inference_module.model_fn(local_model_dir)
print(f"model_fn returned: {type(local_model).__name__}")

csv_payload = test_x.to_csv(index=False, header=False).encode("utf-8")
data = inference_module.input_fn(csv_payload, "text/csv")
prediction = inference_module.predict_fn(data, local_model)
body, response_content_type = inference_module.output_fn(prediction, "text/csv")
local_predictions = np.array(body.strip().split("\n"), dtype=float)

local_test_auc = roc_auc_score(test_y, local_predictions)
print(f"Parsed input shape:    {data.shape}")
print(f"Response content type: {response_content_type}")
print(f"Local test AUC:        {local_test_auc:.4f}  (TestAuc recorded on the version: {provenance.get('TestAuc', 'n/a')})")

---
## 3. Deploy with data capture

The Model Package carries the inference specification, so a `ContainerDefinition` naming the package is enough. You create a `Model`, an `EndpointConfig` with `DataCaptureConfig` (100% of inputs and outputs to S3), and an endpoint.

<div class="alert alert-info">
The endpoint is not public; access is controlled by IAM (<code>sagemaker:InvokeEndpoint</code>). Gets deleted it in the clean-up section.
</div>

In [ ]:
inference_instance_type = "ml.m5.xlarge"
resource_name = f"aim408-from-idea-to-prod-monitored-{time.strftime('%d%H%M%S', time.gmtime())}"
capture_s3_uri = f"s3://{bucket_name}/{bucket_prefix}/data-capture"

deployed_model = Model.create(
    model_name=resource_name,
    primary_container=ContainerDefinition(model_package_name=model_package_arn),
    execution_role_arn=sm_role,
)

endpoint_config = EndpointConfig.create(
    endpoint_config_name=resource_name,
    production_variants=[
        ProductionVariant(
            variant_name="AllTraffic",
            model_name=resource_name,
            initial_instance_count=1,
            instance_type=inference_instance_type,
        )
    ],
    data_capture_config=DataCaptureConfig(
        enable_capture=True,
        initial_sampling_percentage=100,
        destination_s3_uri=capture_s3_uri,
        capture_options=[CaptureOption(capture_mode="Input"), CaptureOption(capture_mode="Output")],
        capture_content_type_header=CaptureContentTypeHeader(csv_content_types=["text/csv"]),
    ),
)

# boto3 for the endpoint: see the SDK gap above.
sm_client.create_endpoint(EndpointName=resource_name, EndpointConfigName=resource_name)
endpoint_name = resource_name

print(f"Creating endpoint {endpoint_name} from {model_package_arn}")
print(f"Data capture destination: {capture_s3_uri}/{endpoint_name}/AllTraffic/")
print("Poll its status in the next cell (this takes a few minutes).")

In [ ]:
# CreateEndpoint returns as soon as the request is accepted, so poll until the
# endpoint reaches a terminal state.
print(f"Polling endpoint: {endpoint_name}")

terminal_states = {"InService", "Failed"}
while True:
    desc = sm_client.describe_endpoint(EndpointName=endpoint_name)
    status = desc["EndpointStatus"]
    print(f"  {time.strftime('%H:%M:%S')} | status={status}")
    if status in terminal_states:
        break
    time.sleep(30)

if status != "InService":
    raise RuntimeError(
        f"Endpoint {endpoint_name} deployment ended with status '{status}': {desc.get('FailureReason')}"
    )

capture = desc["DataCaptureConfig"]
print(f"\nEndpoint {endpoint_name} is InService.")
print(f"Data capture: enabled={capture['EnableCapture']} status={capture['CaptureStatus']} "
      f"sampling={capture['CurrentSamplingPercentage']}% -> {capture['DestinationS3Uri']}")

In [ ]:
display(
    HTML('<b>See the SageMaker <a target="top" href="https://studio-{}.studio.{}.sagemaker.aws/inference-experience/endpoints/details/{}">inference endpoint</a> in the Studio UI</b>'.format(
            domain_id, region, endpoint_name))
)

### Inspect the deployed lineage

The lineage graph now connects the endpoint back to the approved version and the MLflow experiment. Studio can take a minute to add the deployment nodes.

In [ ]:
model_package_group_name = described["ModelPackageGroupName"]
model_package_version = described["ModelPackageVersion"]

lineage_url = (
    f"https://studio-{domain_id}.studio.{region}.sagemaker.aws/"
    f"registry/registered-models/{model_package_group_name}/"
    f"versions/version-{model_package_version}/lineage"
)

display(HTML(
    f'<b>Open the <a target="_blank" href="{lineage_url}">'
    "deployed model lineage</a> in SageMaker Studio</b>"
))

![SageMaker lineage connecting the approved model version to its deployment and endpoint](img/model-version-lineage.png)

---
## 4. Test the endpoint

Send the whole test set in one CSV request. The AUC must match the local run.

In [ ]:
response = runtime_client.invoke_endpoint(
    EndpointName=endpoint_name,
    ContentType="text/csv",
    Accept="text/csv",
    Body=test_x.to_csv(index=False, header=False),
)
predictions = np.array(response["Body"].read().decode("utf-8").strip().split("\n"), dtype=float)

endpoint_test_auc = roc_auc_score(test_y, predictions)
print(f"Endpoint test AUC: {endpoint_test_auc:.4f} | local: {local_test_auc:.4f} | "
      f"max |endpoint - local| = {np.abs(predictions - local_predictions).max():.2e}")
pd.crosstab(index=test_y, columns=np.round(predictions), rownames=["actuals"], colnames=["predictions"])

### Log the test in MLflow

Log the endpoint AUC and confusion matrix to the model's training run, next to its other evidence.

In [ ]:
def plot_confusion_matrix(cm, class_names, title="Confusion matrix", cmap=plt.cm.Blues, normalize=False):
    if normalize:
        cm = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis]
    fig, ax = plt.subplots()
    im = ax.imshow(cm, interpolation="nearest", cmap=cmap)
    ax.figure.colorbar(im, ax=ax)
    ax.set(
        xticks=np.arange(cm.shape[1]), yticks=np.arange(cm.shape[0]), ylim=(cm.shape[0] - 0.5, -0.5),
        xticklabels=class_names, yticklabels=class_names, title=title,
        ylabel="Ground truth label", xlabel="Predicted label",
    )
    plt.setp(ax.get_xticklabels(), rotation=30, ha="right", rotation_mode="anchor")
    thresh = cm.max() / 2.0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, format(cm[i, j], ".2f"), ha="center", va="center",
                    color="white" if cm[i, j] > thresh else "black")
    fig.tight_layout()
    return ax, fig


ax, fig = plot_confusion_matrix(confusion_matrix(test_y, np.round(predictions)), ["no", "yes"])

In [ ]:
mlflow_version = mlflow_version_for_package(mlflow_client, registered_model_name, model_package_arn)
model_run_id = mlflow_client.get_model_version(registered_model_name, mlflow_version).run_id
print(f"MLflow model version {registered_model_name} v{mlflow_version} -> training run {model_run_id}")

with mlflow.start_run(run_id=model_run_id):
    mlflow.log_metric("endpoint_test_auc", endpoint_test_auc)
    mlflow.log_figure(fig, "endpoint_test_confusion_matrix.png")
    mlflow.set_tags({"endpoint_name": endpoint_name, "model_package_arn": model_package_arn})

presigned_url = sm_client.create_presigned_mlflow_app_url(
    Arn=mlflow_arn, ExpiresInSeconds=300, SessionExpirationDurationInSeconds=1800
)["AuthorizedUrl"]
model_run_link = f"{presigned_url.split('/auth')[0]}/#/experiments/{experiment.experiment_id}/runs/{model_run_id}/artifacts?workspace=default"
display(HTML(f'<b>Open the <a target="_blank" href="{presigned_url}">MLflow UI</a> first, then the <a target="_blank" href="{model_run_link}">artifacts of the model\'s run</a>.</b>'))

---
## 5. Send production traffic

- **Ground truth arrives later**, so the labels are set aside, keyed by the `InferenceId` sent with each request.
- **The inputs drift:** `campaign` is pushed up and part of the 30–39 age band moves to 60–69.

In [ ]:
# SIMULATED PRODUCTION DRIFT - for the demonstration only.
rng = np.random.default_rng(42)
production_x = test_x.copy()

# 1. Customers are contacted more often. `campaign` is min-max scaled to [0, 1].
production_x["campaign"] = (production_x["campaign"] * 1.5 + 0.02).clip(upper=1.0)

# 2. An older customer base: 40% of the 30-39 age band moves to 60-69. Both are
#    one-hot columns of the same group, so each record still has exactly one age band.
movers = production_x.index[production_x["age_30-39"] == 1]
movers = rng.choice(movers, size=int(0.4 * len(movers)), replace=False)
production_x.loc[movers, "age_30-39"] = 0
production_x.loc[movers, "age_60-69"] = 1

shifted = ["campaign", "age_30-39", "age_60-69"]
print(pd.DataFrame({"test set": test_x[shifted].mean(), "production traffic": production_x[shifted].mean()}).round(4))

# Ground truth, keyed by the InferenceId sent with each request. Kept apart from
# the traffic, to be joined with the captured predictions later.
inference_ids = [f"inf-{i:06d}" for i in range(len(production_x))]
ground_truth = pd.DataFrame({"inference_id": inference_ids, "target": test_y.astype(int)})
ground_truth.to_csv("tmp/ground_truth.csv", index=False)
print(f"\nGround truth stored for {len(ground_truth)} records")

Send one request per record (3–4 minutes). Transient `ModelError` responses that say *Please retry* are retried with the same `InferenceId`.

In [ ]:
%%time
def invoke(payload, inference_id, attempts=3):
    for attempt in range(1, attempts + 1):
        try:
            response = runtime_client.invoke_endpoint(
                EndpointName=endpoint_name,
                ContentType="text/csv",
                Accept="text/csv",
                Body=payload,
                InferenceId=inference_id,
            )
            return float(response["Body"].read().decode("utf-8").strip())
        except runtime_client.exceptions.ModelError as e:
            if attempt == attempts or "Please retry" not in e.response["Error"]["Message"]:
                raise
            print(f"  transient error on {inference_id}, retrying: {e.response['Error']['Message'][:60]}...")
            time.sleep(2 ** attempt)


live_predictions = np.empty(len(production_x))

print(f"Sending {len(production_x)} requests to {endpoint_name} ...")
for i, (inference_id, row) in enumerate(zip(inference_ids, production_x.itertuples(index=False))):
    live_predictions[i] = invoke(",".join(map(str, row)), inference_id)
    if (i + 1) % 1000 == 0:
        print(f"  {i + 1} requests sent")

print(f"Sent {len(production_x)} requests")
print(f"Test AUC on the (drifted) production traffic: {roc_auc_score(test_y, live_predictions):.4f}")

---
## 6. Read the captured data

Data capture writes JSON Lines under `<destination>/<endpoint>/<variant>/YYYY/MM/DD/HH/`, one record per request:

```json
{"captureData": {"endpointInput":  {"data": "0.02,0.96,...", "encoding": "CSV"},
                 "endpointOutput": {"data": "0.0312",       "encoding": "CSV"}},
 "eventMetadata": {"inferenceId": "inf-000010", "inferenceTime": "2025-...Z"}}
```

Only requests with an `InferenceId` are monitored, so the single endpoint test from section 4 is excluded. The cell waits until all production records have landed.

In [ ]:
capture_prefix = f"{bucket_prefix}/data-capture/{endpoint_name}/AllTraffic"


def decode(part):
    data = part["data"]
    return base64.b64decode(data).decode("utf-8") if part.get("encoding") == "BASE64" else data


def read_capture(bucket, prefix, feature_names):
    """Parse every capture file under `prefix` into one row per captured input line."""
    rows = []
    paginator = s3_client.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get("Contents", []):
            body = s3_client.get_object(Bucket=bucket, Key=obj["Key"])["Body"].read().decode("utf-8")
            for line in body.strip().split("\n"):
                record = json.loads(line)
                meta = record["eventMetadata"]
                inputs = decode(record["captureData"]["endpointInput"]).strip().split("\n")
                outputs = decode(record["captureData"]["endpointOutput"]).strip().split("\n")
                if len(inputs) != len(outputs):
                    print(f"Skipping record {meta.get('eventId')}: {len(inputs)} inputs vs {len(outputs)} outputs")
                    continue
                for raw_input, raw_output in zip(inputs, outputs):
                    rows.append({
                        "inference_id": meta.get("inferenceId", ""),
                        "inference_time": meta.get("inferenceTime", ""),
                        "prediction_proba": float(raw_output),
                        **dict(zip(feature_names, map(float, raw_input.split(",")))),
                    })
    return pd.DataFrame(rows)


print("Waiting for data capture files in S3 ...")
deadline = time.time() + 600
while True:
    all_captured = read_capture(bucket_name, capture_prefix, feature_names)
    # A retried request can be captured twice under the same InferenceId; keep the last.
    captured_df = (all_captured[all_captured["inference_id"] != ""]
                   .drop_duplicates("inference_id", keep="last").reset_index(drop=True))
    print(f"  {time.strftime('%H:%M:%S')} | {len(captured_df)} of {len(production_x)} production records captured "
          f"(+ {len(all_captured) - len(captured_df)} rows from requests without an InferenceId)")
    if len(captured_df) >= len(production_x) or time.time() > deadline:
        break
    time.sleep(30)

captured_df["prediction"] = (captured_df["prediction_proba"] > 0.5).astype(int)
print(f"\nMonitoring {len(captured_df)} captured production records")
captured_df[["inference_id", "inference_time", "prediction_proba", "prediction"] + feature_names[:3]].head()

In [ ]:
# The captured output is what the endpoint returned: check it against the live responses.
live = pd.DataFrame({"inference_id": inference_ids, "live_proba": live_predictions})
check = captured_df.merge(live, on="inference_id")
max_diff = (check["prediction_proba"] - check["live_proba"]).abs().max()
print(f"{len(check)} captured records matched to live responses; max |captured - live| = {max_diff:.2e}")

---
## 7. Data drift with Evidently

Compare the captured inputs (current) with the preprocessing baseline (reference) using [`DataDriftPreset`](https://docs.evidentlyai.com/metrics/preset_data_drift). Reports and drift scores are logged to the workshop's MLflow experiment.

In [ ]:
S3Downloader.download(f"{baseline_s3_url}/baseline.csv", "tmp")
reference_data = pd.read_csv("tmp/baseline.csv", header=None, names=feature_names)
current_data = captured_df[feature_names]

print(f"Reference (baseline): {reference_data.shape}")
print(f"Current (captured):   {current_data.shape}")

In [ ]:
def log_drift_metrics(report_dict):
    """Log the dataset-level drift share and the score of every drifted column to MLflow."""
    for metric in report_dict.get("metrics", []):
        name, value, config = metric.get("metric_name", ""), metric.get("value"), metric.get("config") or {}
        if name.startswith("DriftedColumnsCount") and isinstance(value, dict):
            mlflow.log_metric("drifted_columns_count", float(value.get("count", 0)))
            mlflow.log_metric("drifted_columns_share", float(value.get("share", 0)))
        elif name.startswith("ValueDrift") and config.get("column") and config.get("threshold") is not None:
            try:
                score = float(value)
            except (TypeError, ValueError):
                continue
            if score > float(config["threshold"]):
                mlflow.log_metric(f"drift_score.{config['column']}", score)


def drifted_columns(report_dict):
    """(column, score, threshold) for every column whose drift score exceeded its threshold."""
    hits = []
    for metric in report_dict.get("metrics", []):
        config = metric.get("config") or {}
        if metric.get("metric_name", "").startswith("ValueDrift") and config.get("threshold") is not None:
            try:
                score = float(metric.get("value"))
            except (TypeError, ValueError):
                continue
            if score > float(config["threshold"]):
                hits.append((config["column"], score, float(config["threshold"])))
    return sorted(hits, key=lambda h: h[1], reverse=True)

In [ ]:
os.makedirs("tmp/reports", exist_ok=True)
timestamp_suffix = datetime.now().strftime("%Y%m%d_%H%M%S")
monitoring_tags = {"endpoint_name": endpoint_name, "model_package_arn": model_package_arn, "monitoring": "evidently"}

with mlflow.start_run(run_name=f"monitoring-data-drift-{timestamp_suffix}", tags=monitoring_tags) as drift_run:
    mlflow.log_params({
        "reference_data": f"{baseline_s3_url}/baseline.csv",
        "reference_size": len(reference_data),
        "current_data": f"{capture_s3_uri}/{endpoint_name}/AllTraffic",
        "current_size": len(current_data),
    })

    data_drift_report = Report(metrics=[DataDriftPreset(), DataSummaryPreset()]).run(
        reference_data=Dataset.from_pandas(reference_data, data_definition=DataDefinition()),
        current_data=Dataset.from_pandas(current_data, data_definition=DataDefinition()),
    )
    data_drift_report_file = f"tmp/reports/data_drift_{timestamp_suffix}"
    data_drift_report.save_html(f"{data_drift_report_file}.html")
    data_drift_report.save_json(f"{data_drift_report_file}.json")
    mlflow.log_artifact(f"{data_drift_report_file}.html", "evidently_data_drift")
    mlflow.log_artifact(f"{data_drift_report_file}.json", "evidently_data_drift")

    drift_results = data_drift_report.dict()
    log_drift_metrics(drift_results)

hits = drifted_columns(drift_results)
print(f"Drifted columns: {len(hits)} of {len(feature_names)}")
for column, score, threshold in hits:
    print(f"  {column:<24}score={score:.4f}  threshold={threshold}")
print(f"\nMLflow run: {drift_run.info.run_name} ({drift_run.info.run_id})")

The two shifted features should come out on top. The interactive report:

In [ ]:
IFrame(src=f"{data_drift_report_file}.html", width=1000, height=700)

---
## 8. Model quality with Evidently

Join the ground truth to the captured predictions on `InferenceId` and run [`ClassificationPreset`](https://docs.evidentlyai.com/metrics/preset_classification) (accuracy, precision, recall, F1).

In [ ]:
ground_truth = pd.read_csv("tmp/ground_truth.csv")
evaluation_data = captured_df.merge(ground_truth, on="inference_id", how="inner")

print(f"Captured predictions: {len(captured_df)} | ground truth: {len(ground_truth)} | joined: {len(evaluation_data)}")
print(pd.crosstab(evaluation_data["target"], evaluation_data["prediction"], rownames=["actual"], colnames=["predicted"]))

In [ ]:
def log_classification_metrics(report_dict):
    """Log every scalar metric of the classification report to MLflow (per-label ones suffixed)."""
    for metric in report_dict.get("metrics", []):
        name = metric.get("metric_name", "").split("(")[0].strip()
        value = metric.get("value")
        if isinstance(value, dict):
            for label, v in value.items():
                mlflow.log_metric(f"{name}_label_{label}", float(v))
        elif value is not None:
            try:
                mlflow.log_metric(name, float(value))
            except (TypeError, ValueError):
                pass


with mlflow.start_run(run_name=f"monitoring-model-quality-{timestamp_suffix}", tags=monitoring_tags) as quality_run:
    mlflow.log_params({"evaluated_records": len(evaluation_data), "ground_truth": "tmp/ground_truth.csv"})

    classification_report = Report(metrics=[ClassificationPreset()]).run(
        reference_data=None,
        current_data=Dataset.from_pandas(
            evaluation_data,
            data_definition=DataDefinition(
                classification=[BinaryClassification(target="target", prediction_labels="prediction", pos_label=1)]
            ),
        ),
    )
    classification_report_file = f"tmp/reports/model_quality_{timestamp_suffix}"
    classification_report.save_html(f"{classification_report_file}.html")
    classification_report.save_json(f"{classification_report_file}.json")
    mlflow.log_artifact(f"{classification_report_file}.html", "evidently_model_quality")
    mlflow.log_artifact(f"{classification_report_file}.json", "evidently_model_quality")

    quality_results = classification_report.dict()
    log_classification_metrics(quality_results)

for metric in quality_results["metrics"]:
    name = metric["metric_name"].split("(")[0]
    if name in ("Accuracy", "Precision", "Recall", "F1Score"):
        print(f"  {name:<12}{float(metric['value']):.4f}")
print(f"\nMLflow run: {quality_run.info.run_name} ({quality_run.info.run_id})")

In [ ]:
IFrame(src=f"{classification_report_file}.html", width=1000, height=600)

### View the monitoring runs in MLflow

Both runs are tagged `monitoring = evidently`.

In [ ]:
presigned_url = sm_client.create_presigned_mlflow_app_url(
    Arn=mlflow_arn, ExpiresInSeconds=300, SessionExpirationDurationInSeconds=1800
)["AuthorizedUrl"]
monitoring_runs_link = (
    f"{presigned_url.split('/auth')[0]}/#/experiments/{experiment.experiment_id}/runs"
    f"?workspace=default&searchFilter=tags.monitoring%20%3D%20%22evidently%22"
)
display(HTML(f'<b>Open the <a target="_blank" href="{presigned_url}">MLflow UI</a> first, then the <a target="_blank" href="{monitoring_runs_link}">monitoring runs</a> of the experiment.</b>'))

### Demo: trace a served model back to its training data with Kiro

Because the registry, pipeline, and approval steps record metadata at every hop, a coding agent with read access can answer audit questions about a live endpoint. In the recording, `kiro-cli` answers:

1. **Which dataset trained this endpoint's model?** Endpoint → model → Model Package → training job → pipeline `InputDataUrl`.
2. **Where is the training file in S3?** From the preprocess step's `results.json`.
3. **Who approved and promoted it?** Package metadata plus CloudTrail `UpdateModelPackage` events.

Any agent with the same read access can follow these links.

<video src="videos/modelops-investigation-kiro.mp4" controls style="display: block; width: 75%; margin: 0 auto;" preload="metadata">
  Your browser cannot play this video. <a href="videos/modelops-investigation-kiro.mp4">Download it</a> instead.
</video>

---
## 9. Going further: automate monitoring

In production these checks run on a schedule and alert on thresholds:

- **SageMaker Model Monitor:** [Immersion Day model monitoring lab](https://github.com/aws-samples/amazon-sagemaker-immersion-day/blob/master/classical-ml/lab-4-model-monitoring/model-monitoring.ipynb).
- **Evidently with SageMaker Pipelines and Grafana:** [AIOps on SageMaker AI monitoring samples](https://github.com/aws-samples/sample-aiops-on-amazon-sagemakerai/tree/main/monitoring); `sagemaker-realtime-inference-monitoring-evidently` is closest to this notebook.

---
## Summary

- Deployed only a `production`/`Approved` version, from its inference specification alone.
- Validated the serving code locally, then tested the endpoint and logged the result to MLflow.
- Captured traffic with `InferenceId`s and measured data drift and model quality with Evidently.

### Next steps

- Automate the checks (section 9) and trigger the notebook 03 pipeline on alerts.
- Add prediction drift, business KPIs, and fairness on live traffic.
- Continue with [notebook 05](05-templatize-deployment.ipynb) for staged CI/CD deployment.

## Clean-up

Delete the endpoint (billed per instance-hour), its config, and the model. The registry, MLflow runs, and S3 data are kept.

In [ ]:
for label, delete in (
    ("endpoint", lambda: sm_client.delete_endpoint(EndpointName=endpoint_name)),
    ("endpoint config", lambda: endpoint_config.delete()),
    ("model", lambda: deployed_model.delete()),
):
    try:
        delete()
        print(f"Deleted {label}")
    except Exception as e:
        print(f"Could not delete {label}: {e}")

# Catch anything left over from an earlier run of this notebook.
leftovers = (
    ("endpoint", sm_client.list_endpoints()["Endpoints"], "EndpointName", lambda n: sm_client.delete_endpoint(EndpointName=n)),
    ("endpoint config", sm_client.list_endpoint_configs()["EndpointConfigs"], "EndpointConfigName", lambda n: sm_client.delete_endpoint_config(EndpointConfigName=n)),
    ("model", sm_client.list_models()["Models"], "ModelName", lambda n: sm_client.delete_model(ModelName=n)),
)
for label, items, key, delete in leftovers:
    for item in items:
        name = item[key]
        if name != resource_name and name.startswith("aim408-from-idea-to-prod-monitored"):
            try:
                delete(name)
                print(f"Deleted leftover {label} {name}")
            except Exception as e:
                print(f"Could not delete leftover {label} {name}: {e}")

print("\nClean-up complete. MLflow runs, the Model Registry and the S3 data are preserved.")

## Additional resources
- [Capture data from a real-time endpoint](https://docs.aws.amazon.com/sagemaker/latest/dg/model-monitor-data-capture-endpoint.html)
- [Deploy a model from the Model Registry](https://docs.aws.amazon.com/sagemaker/latest/dg/model-registry-deploy.html)
- [Evidently documentation](https://docs.evidentlyai.com/) and [GitHub repository](https://github.com/evidentlyai/evidently)
- [Amazon SageMaker Model Monitor](https://docs.aws.amazon.com/sagemaker/latest/dg/model-monitor.html)
- [SageMaker Immersion Day: model monitoring lab](https://github.com/aws-samples/amazon-sagemaker-immersion-day/blob/master/classical-ml/lab-4-model-monitoring/model-monitoring.ipynb)
- [AIOps on Amazon SageMaker AI: monitoring samples](https://github.com/aws-samples/sample-aiops-on-amazon-sagemakerai/tree/main/monitoring)
- [Amazon SageMaker AI with MLflow](https://docs.aws.amazon.com/sagemaker/latest/dg/mlflow.html)

# Shutdown kernel

In [ ]:
%%html

<p><b>Shutting down your kernel for this notebook to release resources.</b></p>
<button class="sm-command-button" data-commandlinker-command="kernelmenu:shutdown" style="display:none;">Shutdown Kernel</button>
        
<script>
try {
    els = document.getElementsByClassName("sm-command-button");
    els[0].click();
}
catch(err) {
    // NoOp
}    
</script>